# IAA Hackathon 2026 — Notebook 02: XGBoost Tweedie

Fits Tweedie XGBoost (power=1.5) for motor, buildings, and contents using feature selection from Features_v2.xlsx. Uses time-based split (2021-2023 train, 2024 validate) with log(exposure) as offset. Generates predictions for test files and saves a submission CSV.

In [0]:
# Step 1: Load processed training data, test data, and Features_v3 spreadsheet
import pandas as pd
import numpy as np
import os

%pip install xgboost openpyxl -q

DATA_DIR = '/Workspace/Users/bridget.macdonnell1@gmail.com/iaa-hackathon-2026/data/'
PROC_DIR = os.path.join(DATA_DIR, 'proc')

# Load processed training data (from Notebook 1)
train = {}
for line in ['motor', 'buildings', 'contents']:
    train[line] = pd.read_parquet(os.path.join(PROC_DIR, f'proc_{line}.parquet'))
    print(f'{line}: {train[line].shape} rows, target mean={train[line]["ultimate_claims"].mean():.4f}')

# Load test files
test = {}
for line in ['motor', 'buildings', 'contents']:
    test[line] = pd.read_parquet(os.path.join(DATA_DIR, f'test_{line}.parquet'))
    print(f'test_{line}: {test[line].shape}')

# Load enrichment files
customers = pd.read_parquet(os.path.join(DATA_DIR, 'customers.parquet'))
addresses = pd.read_parquet(os.path.join(DATA_DIR, 'addresses.parquet'))
weather = pd.read_parquet(os.path.join(DATA_DIR, 'weather.parquet'))
sample_submission = pd.read_parquet(os.path.join(DATA_DIR, 'sample_submission.parquet'))

# Load Features_v3 spreadsheet (uniform layout: header at row 1, no skiprows needed)
FEATURES_PATH = '/Workspace/Users/bridget.macdonnell1@gmail.com/iaa-hackathon-2026/notebooks_v4/Features_v3.xlsx'
sheet_map = {'motor': 'Motor', 'buildings': 'Building', 'contents': 'Contents'}
skiprows_map = {'motor': 0, 'buildings': 0, 'contents': 0}

feature_selection = {}
for line, sheet in sheet_map.items():
    df = pd.read_excel(FEATURES_PATH, sheet_name=sheet, skiprows=skiprows_map[line])
    # Clean feature names (strip whitespace)
    df.iloc[:, 0] = df.iloc[:, 0].astype(str).str.strip()
    # Keep only features with Include? == 1 (exclude target)
    kept = df[df.iloc[:, 1] == 1].iloc[:, 0].tolist()
    feature_selection[line] = kept
    print(f'\n{line}: {len(kept)} features selected from Features_v3')
    if 'Year' in kept:
        print(f'  Year included: Yes')

# Trend factors for 2025 predictions (from corrected BF development factors)
# Motor: CAGR 2021-2023 ~3.7% (severity-driven, frequency declining); 2024 still slightly below trend
# Buildings: CAGR 2021-2024 = 9.6% (very consistent upward trend)
# Contents: CAGR 2021-2024 = 4.9% (moderate upward trend)
TREND_FACTORS = {
    'motor': 1.04,
    'buildings': 1.096,
    'contents': 1.049,
}
print(f'\nTrend uplift factors for 2025: {TREND_FACTORS}')

print(f'\nsample_submission: {sample_submission.shape}')
print(f'Row ID prefixes: {sample_submission["row_id"].str[0].value_counts().to_dict()}')

In [0]:
# Step 2: Prepare test data (clean + join enrichment)
# Same cleaning as Notebook 1: exposure, Year, construction_type, customer_id padding

# Clean test files
for line in ['motor', 'buildings', 'contents']:
    df = test[line]
    # Fix exposure
    ps = pd.to_datetime(df['period_start'])
    pe = pd.to_datetime(df['period_end'])
    df['exposure'] = ((pe - ps).dt.days + 1) / 365.0
    # Create Year
    df['Year'] = ps.dt.year
    # Strip IDs
    for col in ['policy_id', 'customer_id', 'address_id', 'row_id']:
        if col in df.columns:
            df[col] = df[col].astype(str).str.strip()
    print(f'test_{line}: exposure mean={df["exposure"].mean():.4f}, Year={df["Year"].unique()}')

# Fix construction_type in test_buildings
df = test['buildings']
df['construction_type'] = df['construction_type'].astype(str).str.strip().str.upper()
df['construction_type'] = df['construction_type'].replace({'W': 'TIMBER', 'T': 'TIMBER'})
print(f'test_buildings construction_type: {df["construction_type"].value_counts().to_dict()}')

# Zero-pad motor customer_id
test['motor']['customer_id'] = test['motor']['customer_id'].str.zfill(8)
customers['customer_id'] = customers['customer_id'].astype(str).str.zfill(8)

# Drop compulsory_excess if present (was constant in training, dropped in Notebook 1)
for line in ['motor', 'buildings', 'contents']:
    if 'compulsory_excess' in test[line].columns:
        test[line].drop('compulsory_excess', axis=1, inplace=True)

# Prepare enrichment joins
customers_clean = customers.drop(columns=['surname', 'date_of_birth'], errors='ignore')
weather_agg = weather.groupby(['address_id', 'year'], as_index=False).mean(numeric_only=True)
weather_agg.rename(columns={'year': 'Year'}, inplace=True)

# Join enrichment to test files
for line in ['motor', 'buildings', 'contents']:
    df = test[line]
    n_before = len(df)
    df = df.merge(customers_clean, on='customer_id', how='left', suffixes=('', '_cust'))
    df = df.merge(addresses, on='address_id', how='left', suffixes=('', '_addr'))
    df = df.merge(weather_agg, on=['address_id', 'Year'], how='left', suffixes=('', '_weather'))
    test[line] = df
    print(f'test_{line}: {n_before} -> {len(df)} rows, {df.shape[1]} cols')
    # Check weather match for 2025
    weather_match = df['hail_days'].notna().sum()
    print(f'  Weather matched: {weather_match}/{len(df)} ({100*weather_match/len(df):.1f}%)')

In [0]:
# Step 3: Fit XGBoost Tweedie for all 3 lines
# Time-based split: 2021-2023 train, 2024 validate
# Offset: log(exposure) as base_margin
# No k-fold, no hypertuning (per plan)

import xgboost as xgb

def tweedie_deviance(y, y_pred, p=1.5):
    y = np.array(y, dtype=float)
    y_pred = np.maximum(y_pred, 1e-20)  # avoid log(0)
    return 2 * (np.power(np.maximum(y, 0), 2-p) / ((1-p)*(2-p))
                - y * np.power(y_pred, 1-p) / (1-p)
                + np.power(y_pred, 2-p) / (2-p))

def tweedie_d2(y, y_pred, p=1.5):
    dev = tweedie_deviance(y, y_pred, p).sum()
    dev_null = tweedie_deviance(y, np.mean(y), p).sum()
    return 1 - dev / dev_null

def prepare_features(train_df, test_df, feature_list):
    """Select features, label-encode categoricals, return X_train, X_test ready for XGBoost."""
    # Only keep features that exist in the data
    available = [f for f in feature_list if f in train_df.columns and f not in ['ultimate_claims', 'row_id', 'policy_id', 'customer_id', 'address_id', 'period_start', 'period_end']]
    print(f'  Available features: {len(available)} / {len(feature_list)}')
    missing = [f for f in feature_list if f not in train_df.columns]
    if missing:
        print(f'  Missing from data: {missing}')
    
    X_train = train_df[available].copy()
    X_test = test_df[available].copy() if test_df is not None else None
    
    # Encode categoricals (object or bool)
    for col in available:
        if X_train[col].dtype == 'object' or X_train[col].dtype == 'bool':
            cats = pd.Categorical(X_train[col])
            X_train[col] = cats.codes.astype('float64')
            if X_test is not None:
                X_test[col] = pd.Categorical(X_test[col], categories=cats.categories).codes.astype('float64')
    
    return X_train, X_test, available

# XGBoost parameters (from v3, well-regularised)
XGB_PARAMS = {
    'objective': 'reg:tweedie',
    'tweedie_variance_power': 1.5,
    'max_depth': 6,
    'eta': 0.1,
    'subsample': 0.8,
    'colsample_bytree': 0.5,
    'min_child_weight': 10,
    'reg_alpha': 0.5,
    'reg_lambda': 2.0,
    'seed': 42,
    'n_jobs': -1,
    'verbosity': 0,
}

models = {}
val_scores = {}
feature_importances = {}
test_predictions = {}

for line in ['motor', 'buildings', 'contents']:
    print(f'\n{"="*60}')
    print(f'=== {line.upper()} XGBoost ===')
    print(f'{"="*60}')
    
    train_df = train[line]
    test_df = test[line]
    
    # Prepare features
    X_train, X_test, feat_cols = prepare_features(train_df, test_df, feature_selection[line])
    y_train = train_df['ultimate_claims'].values
    exposure_train = train_df['exposure'].values
    
    # Time-based split: 2021-2023 train, 2024 validate
    year = train_df['Year'].values
    train_mask = year <= 2023
    val_mask = year == 2024
    
    X_tr = X_train[train_mask]
    y_tr = y_train[train_mask]
    exp_tr = exposure_train[train_mask]
    
    X_val = X_train[val_mask]
    y_val = y_train[val_mask]
    exp_val = exposure_train[val_mask]
    
    print(f'  Train: {len(X_tr)} rows, Val: {len(X_val)} rows')
    print(f'  Train target mean: {y_tr.mean():.4f}, Val target mean: {y_val.mean():.4f}')
    
    exp_test = test_df['exposure'].values
    
    # Use DMatrix API with base_margin = log(exposure) for BOTH training and prediction
    # Model learns log claim rate, offset scales prediction by exposure
    dtrain = xgb.DMatrix(X_tr, label=y_tr, base_margin=np.log(exp_tr + 1e-10), weight=exp_tr)
    dval = xgb.DMatrix(X_val, label=y_val, base_margin=np.log(exp_val + 1e-10), weight=exp_val)
    dtest = xgb.DMatrix(X_test, base_margin=np.log(exp_test + 1e-10))
    
    model = xgb.train(XGB_PARAMS, dtrain, num_boost_round=300, evals=[(dval, 'val')], early_stopping_rounds=50, verbose_eval=False)
    models[line] = model
    
    # Validate
    val_pred = model.predict(dval)
    val_pred = np.maximum(val_pred, 0)
    d2 = tweedie_d2(y_val, val_pred, p=1.5)
    val_scores[line] = d2
    print(f'  Validation D2 (Tweedie p=1.5): {d2:.4f}')
    print(f'  Val pred mean: {val_pred.mean():.4f} (actual mean: {y_val.mean():.4f})')
    
    # Feature importance
    scores = model.get_score(importance_type='gain')
    imp = pd.DataFrame({'feature': feat_cols, 'importance': [scores.get(f, 0) for f in feat_cols]})
    imp = imp.sort_values('importance', ascending=False)
    feature_importances[line] = imp
    print(f'  Top 10 features:')
    print(f'  {imp.head(10).to_string(index=False)}')
    
    # Predict on test
    test_pred = model.predict(dtest)
    test_pred = np.maximum(test_pred, 0.01)
    test_predictions[line] = test_pred
    print(f'  Test predictions: mean={test_pred.mean():.4f}, min={test_pred.min():.4f}, max={test_pred.max():.4f}')

print(f'\n{"="*60}')
print(f'=== Summary: Validation D2 Scores ===')
for line in ['motor', 'buildings', 'contents']:
    print(f'  {line}: {val_scores[line]:.4f}')
print(f'  Average: {np.mean(list(val_scores.values())):.4f}')

In [0]:
# Step 4: Generate submission file
SUB_DIR = '/Workspace/Users/bridget.macdonnell1@gmail.com/iaa-hackathon-2026/submissions_v4/'
os.makedirs(SUB_DIR, exist_ok=True)

# Build submission from sample_submission
submission = sample_submission.copy()

# Match predictions to row_ids
for line in ['motor', 'buildings', 'contents']:
    test_df = test[line]
    preds = test_predictions[line]
    # Apply trend uplift for 2025 (trees can't extrapolate, so explicit inflation needed)
    preds = preds * TREND_FACTORS[line]
    # Create mapping: row_id -> prediction
    pred_map = dict(zip(test_df['row_id'], preds))
    # Apply to submission
    mask = submission['row_id'].isin(pred_map)
    submission.loc[mask, 'prediction'] = submission.loc[mask, 'row_id'].map(pred_map)
    print(f'{line}: {mask.sum()} predictions added (trend factor={TREND_FACTORS[line]})')

# Floor any remaining zeros to 0.01
submission['prediction'] = submission['prediction'].clip(lower=0.01)

# Save
sub_path = os.path.join(SUB_DIR, 'xgboost_tweedie_v4.csv')
submission.to_csv(sub_path, index=False)
print(f'\nSaved: {sub_path} ({len(submission)} rows)')
print(f'Prediction stats: mean={submission["prediction"].mean():.4f}, min={submission["prediction"].min():.4f}, max={submission["prediction"].max():.4f}')
print(f'By line: M={submission[submission["row_id"].str.startswith("M")]["prediction"].mean():.4f}, '
      f'B={submission[submission["row_id"].str.startswith("B")]["prediction"].mean():.4f}, '
      f'C={submission[submission["row_id"].str.startswith("C")]["prediction"].mean():.4f}')

In [0]:
# Step 5: Excel checkpoint — save D2 scores and feature importance
EXCEL_DIR = '/Workspace/Users/bridget.macdonnell1@gmail.com/iaa-hackathon-2026/Excel files_v4/'

with pd.ExcelWriter(os.path.join(EXCEL_DIR, 'xgboost_v4_results.xlsx'), engine='openpyxl') as writer:
    # Sheet 1: D2 Scores
    scores_df = pd.DataFrame([
        {'line': line, 'd2_tweedie_1.5': val_scores[line]} 
        for line in ['motor', 'buildings', 'contents']
    ])
    scores_df.loc['average'] = ['average', scores_df['d2_tweedie_1.5'].mean()]
    scores_df.to_excel(writer, sheet_name='D2 Scores', index=False)
    
    # Sheets 2-4: Feature importance per line
    for line in ['motor', 'buildings', 'contents']:
        feature_importances[line].to_excel(writer, sheet_name=f'{line} importance', index=False)

print(f'Saved: {EXCEL_DIR}xgboost_v4_results.xlsx')
print(f'Sheets: D2 Scores, motor importance, buildings importance, contents importance')
print(f'\nValidation D2 scores:')
for line in ['motor', 'buildings', 'contents']:
    print(f'  {line}: {val_scores[line]:.4f}')
print(f'  Average: {np.mean(list(val_scores.values())):.4f}')

In [0]:
# Trend analysis: claims_rate by year for each line
# This determines the inflation/trend uplift to apply for 2025 predictions

print('=== Year-on-Year Trend Analysis ===\n')

trend_factors = {}
for line in ['motor', 'buildings', 'contents']:
    df = train[line]
    by_year = df.groupby('Year').agg(
        exposure=('exposure', 'sum'),
        ultimate=('ultimate_claims', 'sum'),
        claims=('claim_count', 'sum') if 'claim_count' in df.columns else ('ultimate_claims', 'count'),
    ).reset_index()
    by_year['claims_rate'] = by_year['ultimate'] / by_year['exposure']
    if 'claim_count' in df.columns:
        by_year['frequency'] = df.groupby('Year')['claim_count'].sum().values / by_year['exposure']
        by_year['severity'] = by_year['ultimate'] / (df.groupby('Year')['claim_count'].sum().values)
    
    print(f'--- {line.upper()} ---')
    print(by_year[['Year', 'exposure', 'claims_rate']].to_string(index=False))
    
    # Calculate year-on-year change in claims_rate
    rates = by_year['claims_rate'].values
    years = by_year['Year'].values
    yoy_changes = []
    for i in range(1, len(rates)):
        if rates[i-1] > 0:
            change = (rates[i] / rates[i-1] - 1)
            yoy_changes.append(change)
            print(f'  {years[i-1]}->{years[i]}: {change*100:+.2f}%')
    
    # Average annual trend (geometric mean of YoY changes, or simple linear regression)
    if len(yoy_changes) > 0:
        avg_trend = np.mean(yoy_changes)
        # Also compute linear trend: (rate_2024 - rate_2021) / 3 years
        linear_trend = (rates[-1] / rates[0]) ** (1/(len(rates)-1)) - 1
        print(f'  Average YoY change: {avg_trend*100:.2f}%')
        print(f'  CAGR ({years[0]}-{years[-1]}): {linear_trend*100:.2f}%')
        trend_factors[line] = linear_trend  # use CAGR as the trend factor
    print()

print('=== Trend factors to apply for 2025 ===')
for line, trend in trend_factors.items():
    # For 2025, apply 1 year of trend on top of 2024
    uplift = 1 + trend
    print(f'  {line}: CAGR={trend*100:.2f}%, 2025 uplift factor = {uplift:.4f}')

In [0]:
# Step 6: Save XGBoost validation predictions for use in Notebook 3 (GLM & Blend)

for line in ['motor', 'buildings', 'contents']:
    model = models[line]
    train_df = train[line]
    feat_cols = [f for f in feature_selection[line] if f in train_df.columns and f not in ['ultimate_claims', 'row_id', 'policy_id', 'customer_id', 'address_id', 'period_start', 'period_end']]
    
    year = train_df['Year'].values
    val_mask = year == 2024
    X_val = train_df.loc[val_mask, feat_cols].copy()
    
    for col in feat_cols:
        if X_val[col].dtype == 'object' or X_val[col].dtype == 'bool':
            cats = pd.Categorical(train_df[col])
            X_val[col] = pd.Categorical(X_val[col], categories=cats.categories).codes.astype('float64')
    
    exp_val = train_df.loc[val_mask, 'exposure'].values
    y_val = train_df.loc[val_mask, 'ultimate_claims'].values
    row_ids_val = train_df.loc[val_mask, 'row_id'].values
    
    dval = xgb.DMatrix(X_val, base_margin=np.log(exp_val + 1e-10))
    val_pred = model.predict(dval)
    val_pred = np.maximum(val_pred, 0)
    
    val_df = pd.DataFrame({'row_id': row_ids_val, 'xgb_pred': val_pred, 'ultimate_claims': y_val, 'exposure': exp_val})
    val_df.to_parquet(os.path.join(PROC_DIR, f'xgb_val_pred_{line}.parquet'), index=False)
    print(f'{line}: saved {len(val_df)} validation predictions')

print('\nValidation predictions saved to data/proc/ for use in Notebook 3 blend')

## Step 6: AI/Human Decision Log

### What AI did
- Loaded processed data from Notebook 1 (proc_motor, proc_buildings, proc_contents)
- Loaded and cleaned test data (exposure, Year, construction_type, customer_id padding)
- Joined customers, addresses, weather to test files (weather matched for 2025)
- Read Features_v2.xlsx for feature selection (motor 64, buildings 59, contents 58)
- Label-encoded categorical features for XGBoost
- Fitted XGBoost reg:tweedie (power=1.5) with sample_weight=exposure and base_margin=log(exposure)
- Used time-based split (2021-2023 train, 2024 validate) — no k-fold
- Used v3 parameters: max_depth=6, colsample_bytree=0.5, min_child_weight=10, reg_alpha=0.5, reg_lambda=2.0
- Calculated Tweedie D² scores on 2024 validation
- Generated test predictions (floored at 0.01)
- Saved submission CSV and Excel checkpoint

### What human decided
- Feature selection via Features_v2 spreadsheet (user judgement on what to include/exclude)
- Skip k-fold and hypertuning for speed (user instruction)
- Use time-based split (user instruction)
- Floor zeros to 0.01 (user instruction)

### Key decisions
1. **Sample weight = exposure**: Using exposure as sample weight so the model focuses on rows with more exposure
2. **Base margin = log(exposure)**: Offset so the model predicts claim rate, then multiplied by exposure
3. **No hypertuning**: Using v3 parameters that worked well — saves time for the hackathon
4. **Label encoding**: For categoricals, using factorize (not one-hot) to avoid dimensionality blow-up

### Future improvements
- Hyperparameter tuning (if time permits after initial submission)
- K-fold cross-validation for more robust validation
- Better categorical encoding (target encoding, CatBoost)
- Large claims modelling (separate treatment for motor large claims)